## Etape 1 : construction des features RFM

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


transactions = pd.read_csv("../Lumina & Co - CRM/transactions.csv")
customers = pd.read_csv("../Lumina & Co - CRM/customers.csv")

transactions.shape, customers.shape

**A retenir :** `transactions.shape` et `customers.shape` donnent (nb_lignes, nb_colonnes) de chaque fichier. Ca sert juste a verifier que le chargement a fonctionne avant d'aller plus loin.

In [ ]:
# enleve les lignes sans customer_id et calcule le montant de chaque ligne
transactions = transactions[transactions['customer_id'].notna()]
transactions['line_total'] = transactions['quantity'] * transactions['unit_price']

In [ ]:
# date de reference = derniere date du dataset (pas aujourd'hui)
transactions['invoice_date'] = pd.to_datetime(transactions['invoice_date'])
snapshot_date = transactions['invoice_date'].max()
snapshot_date

**A retenir :** `snapshot_date` est la date la plus recente presente dans les transactions. C'est la date de reference utilisee pour calculer la recence de chaque client (pas la date d'aujourd'hui, sinon la recence serait faussee par le temps ecoule depuis l'export des donnees).

In [ ]:
# recence = nb de jours depuis le dernier achat de chaque client
recency = transactions.groupby('customer_id')['invoice_date'].max()
recency = (snapshot_date - recency).dt.days
recency.name = 'recency'

**A retenir :** `snapshot_date` est la date la plus recente presente dans les transactions. C'est la date de reference utilisee pour calculer la recence de chaque client (pas la date d'aujourd'hui, sinon la recence serait faussee par le temps ecoule depuis l'export des donnees).

In [ ]:
# frequence = nb de factures distinctes par client
frequency = transactions.groupby('customer_id')['invoice_id'].nunique()
frequency.name = 'frequency'

In [ ]:
# montant = somme des achats par client
monetary = transactions.groupby('customer_id')['line_total'].sum()
monetary.name = 'monetary'

In [ ]:
# regroupe les 3 mesures dans un seul dataframe
rfm = pd.concat([recency, frequency, monetary], axis=1).reset_index()
rfm.head()

**A retenir :** `rfm` contient maintenant une ligne par client avec ses 3 mesures brutes : `recency` (jours depuis le dernier achat), `frequency` (nb de commandes), `monetary` (montant total depense).

In [ ]:
# verifie le nombre de clients dans le dataframe rfm
rfm.shape

**A retenir :** confirme le nombre total de clients dans la base (une ligne par client) et le nombre de colonnes du dataframe `rfm`.

## Etape 2 : scoring RFM et segments

In [ ]:
# score de 1 a 5 par quintile, recency inversee (recent = score 5)
rfm['r_score'] = pd.qcut(rfm['recency'].rank(method='first'), 5, labels=[5, 4, 3, 2, 1])
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['m_score'] = pd.qcut(rfm['monetary'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])

rfm.head()

**A retenir :** chaque client recoit maintenant 3 notes de 1 (mauvais) a 5 (excellent) sur la recence, la frequence et le montant, calculees par quintile (20% des clients par tranche).

In [ ]:
# concatene les 3 scores pour avoir un code du type 555, 111, etc
rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)
rfm.head()

**A retenir :** `rfm_score` combine les 3 notes en un seul code a 3 chiffres, ex: `"555"` = excellent sur les 3 criteres, `"111"` = mauvais sur les 3.

In [ ]:
# combien de champions (555) et quelle part du CA total
champions = rfm[rfm['rfm_score'] == '555']
nb_champions = len(champions)
part_champions = nb_champions / len(rfm) * 100
part_ca_champions = champions['monetary'].sum() / rfm['monetary'].sum() * 100

nb_champions, part_champions, part_ca_champions

**A retenir :** `nb_champions` = nombre de clients avec le score parfait 555, `part_champions` = leur % dans la base totale, `part_ca_champions` = leur % du chiffre d'affaires total. Compare les deux derniers chiffres : si leur part du CA est bien plus grande que leur part en nombre de clients, ce sont des clients a proteger en priorite.

In [ ]:
# clients a risque : forte valeur/frequence historique mais faible recence
a_risque = rfm[(rfm['r_score'].astype(int) <= 2) & (rfm['f_score'].astype(int) >= 4) & (rfm['m_score'].astype(int) >= 4)]
len(a_risque)

**A retenir :** ces clients avaient une forte valeur et frequence d'achat, mais n'achetent plus recemment (recency faible score). Ce sont d'anciens bons clients en train de partir.

In [ ]:
# clients perdus (111) et leur proportion
perdus = rfm[rfm['rfm_score'] == '111']
len(perdus), len(perdus) / len(rfm) * 100

**A retenir :** les clients "111" sont mauvais sur les 3 criteres : recence, frequence et montant. Le 2e chiffre est leur proportion en % dans la base totale.

## Visualisation des scores

In [ ]:
# matrice RFM : nombre de clients pour chaque combinaison recence x frequence
matrice = rfm.pivot_table(index='f_score', columns='r_score', values='customer_id', aggfunc='count')
matrice = matrice.sort_index(axis=0).sort_index(axis=1)

plt.figure(figsize=(6, 5))
plt.imshow(matrice, cmap='Blues')
plt.colorbar(label='nombre de clients')
plt.xlabel('score recence')
plt.ylabel('score frequence')
plt.xticks(range(len(matrice.columns)), matrice.columns)
plt.yticks(range(len(matrice.index)), matrice.index)
plt.title('Matrice RFM (frequence x recence)')
plt.gca().invert_yaxis()
plt.show()

**A retenir :** chaque case de la matrice = nombre de clients pour une combinaison (score frequence, score recence) donnee. Plus une case est foncee, plus il y a de clients dedans. En bas a gauche (frequence faible, recence faible = achat ancien et rare) et en haut a droite (frequence forte, recence forte = achat frequent et recent) sont les deux zones les plus interessantes a regarder.

In [ ]:
# panier moyen = montant total / nombre de commandes
rfm['avg_basket'] = rfm['monetary'] / rfm['frequency']
rfm['avg_basket'].describe()

**A retenir :** `avg_basket` = ce que depense un client en moyenne a chaque commande. `.describe()` donne la moyenne, l'ecart-type, le min, le max et les quartiles de cette colonne sur toute la base.

In [ ]:
# distribution du panier moyen (echelle log car quelques clients B2B avec un panier tres eleve)
plt.figure(figsize=(7, 4))
plt.hist(rfm['avg_basket'], bins=50, color='#4C72B0')
plt.xscale('log')
plt.xlabel('panier moyen (log)')
plt.ylabel('nombre de clients')
plt.title('Distribution du panier moyen par client')
plt.show()

**A retenir :** l'histogramme montre que la grande majorite des clients ont un panier moyen concentre sur une petite plage de valeurs (le gros pic a gauche), avec une longue traine de quelques clients a panier tres eleve (probablement des B2B, comme souligne dans le TP1) qui etirent le graphique vers la droite.